# MALTO — v8: LightGBM Stacking (ModernBERT OOF + SVC OOF + Stylometric)

**Goal:** Avoid ALL threshold/temperature post-processing overfitting.

**Key insight:** Instead of optimizing thresholds on OOF (which overfits),
train a LightGBM meta-learner that *learns* the optimal combination of:
- ModernBERT-base softmax probabilities (6 features)
- SVC softmax probabilities (6 features)
- Stylometric features (21 features)
= 33 features total, trained with 5-fold CV on 2400 samples.

**The meta-learner generalizes better than hand-tuned thresholds because:**
- It's trained with proper cross-validation
- It can learn non-linear class-specific relationships
- No manual threshold grid search needed

**Runtime:** ~120 min on T4×2


In [ ]:
import os, warnings, gc, time, re
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_scheduler

from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack as sparse_hstack
import lightgbm as lgb
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

SEED = 42
NUM_LABELS = 6
LABEL_MAP  = {0: 'Human', 1: 'DeepSeek', 2: 'Grok', 3: 'Claude', 4: 'Gemini', 5: 'ChatGPT'}
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda'); torch.cuda.manual_seed_all(SEED)
    N_GPUS = torch.cuda.device_count(); USE_AMP = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32       = False
    print(f'GPU: {torch.cuda.get_device_name(0)} x {N_GPUS}')
else:
    DEVICE = torch.device('cpu'); N_GPUS = 1; USE_AMP = False

KAGGLE_PATHS = [
    '/kaggle/input/malto-recruitment-hackathon',
    '/kaggle/input/test-and-train',
    '/kaggle/input/datasets/aliivaezii/test-and-train',
    '/kaggle/input', '.', '..'
]
DATA_DIR = None
for p in KAGGLE_PATHS:
    if os.path.exists(os.path.join(p, 'train.csv')):
        DATA_DIR = p; break
if DATA_DIR is None and os.path.isdir('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'train.csv' in files:
            DATA_DIR = root; break
assert DATA_DIR is not None

train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
train_df['TEXT'] = train_df['TEXT'].fillna('empty')
test_df['TEXT']  = test_df['TEXT'].fillna('empty')

y_all       = train_df['LABEL'].values
texts_train = train_df['TEXT'].values
texts_test  = test_df['TEXT'].values
n_train, n_test = len(texts_train), len(texts_test)

print(f'Device: {DEVICE} | GPUs: {N_GPUS}')
print(f'Train: {n_train}, Test: {n_test}')
t_start = time.time()


In [ ]:
# ============================================================
# CONFIG
# ============================================================
MODEL_NAME  = 'answerdotai/ModernBERT-base'
MAX_LEN     = 512
BATCH_SIZE  = 16 * N_GPUS
EPOCHS      = 10
LR          = 3e-5
PATIENCE    = 2
N_FOLDS     = 3         # 3-fold neural (fast), 5-fold for SVC/meta
NUM_WORKERS = 4
DRW_CAP     = 20
TOP_K_CKPTS = 3

print(f'Model:   {MODEL_NAME}')
print(f'Batch:   {BATCH_SIZE} ({BATCH_SIZE//N_GPUS}/GPU)  |  Folds: {N_FOLDS}  |  DRW cap: {DRW_CAP}x')


In [ ]:
# ============================================================
# SHARED UTILITIES  (LDAM loss, TextDataset, optimizer)
# ============================================================
class LDAMLoss(nn.Module):
    def __init__(self, class_counts, max_margin=0.5,
                 drw_epoch=2, drw_ramp_epochs=3, label_smoothing=0.1):
        super().__init__()
        safe = np.maximum(class_counts, 1).astype(np.float64)
        margins = np.clip(max_margin / np.power(safe, 0.25), 0, 1.0)
        self.register_buffer('margins', torch.tensor(margins, dtype=torch.float32))
        self.drw_epoch = drw_epoch; self.drw_ramp_epochs = drw_ramp_epochs
        self.current_epoch = 0; self.label_smoothing = label_smoothing
        inv = 1.0 / safe; w = inv / inv.sum() * len(class_counts)
        w   = np.clip(w, w.min(), w.min() * DRW_CAP)
        w   = w / w.sum() * len(class_counts)
        w   = np.nan_to_num(w, nan=1.0, posinf=1.0, neginf=1.0)
        self.register_buffer('drw_weights', torch.tensor(w, dtype=torch.float32))

    def set_epoch(self, e): self.current_epoch = e

    def _blended_weight(self, device):
        ep = self.current_epoch
        if ep < self.drw_epoch: return None
        t = min((ep - self.drw_epoch) / max(self.drw_ramp_epochs, 1), 1.0)
        return (torch.ones_like(self.drw_weights) + t * (self.drw_weights - 1)).to(device)

    def forward(self, logits, targets):
        logits = logits.float()
        if torch.isnan(logits).any(): logits = torch.nan_to_num(logits, nan=0.0)
        margins  = self.margins.to(logits.device)[targets]
        adjusted = logits.clone()
        adjusted[torch.arange(len(targets), device=logits.device), targets] -= margins
        weight = self._blended_weight(logits.device)
        loss   = F.cross_entropy(adjusted, targets, weight=weight,
                                 label_smoothing=self.label_smoothing)
        if torch.isnan(loss): return F.cross_entropy(logits, targets)
        return loss


class TextDataset(Dataset):
    def __init__(self, enc, labels=None, indices=None):
        self.enc = enc; self.labels = labels; self.indices = indices
    def __len__(self):
        return len(self.indices) if self.indices is not None else len(self.enc['input_ids'])
    def __getitem__(self, idx):
        i = self.indices[idx] if self.indices is not None else idx
        item = {k: v[i] for k, v in self.enc.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[i], dtype=torch.long)
        return item


def get_optimizer(model, lr, llrd=0.9):
    no_decay = ['bias', 'LayerNorm', 'layernorm', 'layer_norm']
    params   = []
    head = [(n, p) for n, p in model.named_parameters()
            if 'classifier' in n or 'head' in n]
    if head:
        params += [{'params': [p for n, p in head if not any(nd in n for nd in no_decay)],
                    'lr': lr, 'weight_decay': 0.01},
                   {'params': [p for n, p in head if     any(nd in n for nd in no_decay)],
                    'lr': lr, 'weight_decay': 0.0}]
    layer_nums = set()
    for n, _ in model.named_parameters():
        m = re.search(r'(?:encoder\.layer|layers)\.(\d+)\.', n)
        if m: layer_nums.add(int(m.group(1)))
    num_layers = max(layer_nums) + 1 if layer_nums else 22
    for i in range(num_layers - 1, -1, -1):
        layer_lr = lr * (llrd ** (num_layers - 1 - i))
        lp = [(n, p) for n, p in model.named_parameters()
              if f'encoder.layer.{i}.' in n or f'layers.{i}.' in n]
        if lp:
            params += [{'params': [p for n, p in lp if not any(nd in n for nd in no_decay)],
                        'lr': layer_lr, 'weight_decay': 0.01},
                       {'params': [p for n, p in lp if     any(nd in n for nd in no_decay)],
                        'lr': layer_lr, 'weight_decay': 0.0}]
    assigned = set(id(p) for g in params for p in g['params'])
    rem = [p for n, p in model.named_parameters() if id(p) not in assigned and p.requires_grad]
    if rem: params.append({'params': rem, 'lr': lr * 0.5, 'weight_decay': 0.01})
    return torch.optim.AdamW(params)

print('Utilities ready.')


In [ ]:
# ============================================================
# TOKENIZATION
# ============================================================
print(f'Tokenizing with {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_enc = tokenizer(list(texts_train), max_length=MAX_LEN, padding='max_length',
                      truncation=True, return_tensors='pt')
test_enc  = tokenizer(list(texts_test),  max_length=MAX_LEN, padding='max_length',
                      truncation=True, return_tensors='pt')
print(f'Train: {train_enc["input_ids"].shape}, Test: {test_enc["input_ids"].shape}')


In [ ]:
# ============================================================
# STAGE 1: ModernBERT-base 3-fold → OOF softmax probs
# ============================================================
def train_fold_mb(fold_idx, train_idx, val_idx):
    gc.collect(); torch.cuda.empty_cache()
    print(f'\n--- MB Fold {fold_idx+1}/{N_FOLDS} ---')

    train_loader = DataLoader(TextDataset(train_enc, y_all, train_idx),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
        pin_memory=True, drop_last=True)
    val_loader  = DataLoader(TextDataset(train_enc, y_all, val_idx),
        batch_size=BATCH_SIZE*2, num_workers=NUM_WORKERS, pin_memory=True)
    test_loader = DataLoader(TextDataset(test_enc),
        batch_size=BATCH_SIZE*2, num_workers=NUM_WORKERS, pin_memory=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS, ignore_mismatched_sizes=True)
    base = model.model if hasattr(model, 'model') else model
    base.gradient_checkpointing_enable()
    if N_GPUS > 1: model = nn.DataParallel(model)
    model.to(DEVICE)

    fold_counts = np.bincount(y_all[train_idx], minlength=NUM_LABELS).astype(float)
    criterion   = LDAMLoss(fold_counts); criterion.to(DEVICE)
    optimizer   = get_optimizer(model.module if hasattr(model, 'module') else model, lr=LR)
    total_steps = len(train_loader) * EPOCHS
    scheduler   = get_scheduler('cosine', optimizer,
        num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)
    scaler      = GradScaler(enabled=USE_AMP)

    top_ckpts = []; best_f1 = 0; patience_ctr = 0

    for epoch in range(EPOCHS):
        t0 = time.time(); criterion.set_epoch(epoch)
        model.train(); total_loss, valid_steps = 0.0, 0
        for batch in tqdm(train_loader, desc=f'E{epoch+1}', leave=False):
            optimizer.zero_grad()
            inputs = {k: v.to(DEVICE) for k, v in batch.items()}
            labels = inputs.pop('labels')
            with autocast(enabled=USE_AMP):
                out  = model(**inputs)
                loss = criterion(out.logits, labels)
            if torch.isnan(loss) or torch.isinf(loss):
                scheduler.step(); continue
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            total_loss += loss.item(); valid_steps += 1
            scheduler.step()

        model.eval(); preds, lbls = [], []
        with torch.no_grad():
            for batch in val_loader:
                inp = {k: v.to(DEVICE) for k, v in batch.items()}; lbl = inp.pop('labels')
                with autocast(enabled=USE_AMP): out = model(**inp)
                preds.extend(out.logits.argmax(-1).cpu().numpy())
                lbls.extend(lbl.cpu().numpy())

        val_f1 = f1_score(lbls, preds, average='macro')
        cur_m  = model.module if hasattr(model, 'module') else model
        state  = {k: v.cpu().clone() for k, v in cur_m.state_dict().items()}
        if len(top_ckpts) < TOP_K_CKPTS or val_f1 > top_ckpts[-1][0]:
            if len(top_ckpts) >= TOP_K_CKPTS: top_ckpts.pop()
            top_ckpts.append((val_f1, state)); top_ckpts.sort(key=lambda x: x[0], reverse=True)
        if val_f1 > best_f1:
            best_f1 = val_f1; patience_ctr = 0; marker = ' **'
        else:
            patience_ctr += 1; marker = f' (p={patience_ctr})'
        print(f'  E{epoch+1} | loss={total_loss/max(valid_steps,1):.4f} | f1={val_f1:.4f} | {time.time()-t0:.0f}s{marker}')
        if patience_ctr >= PATIENCE: break

    n_avg = len(top_ckpts)
    avg_state = {k: (sum(c[k].float() for _,c in top_ckpts)/n_avg
                     ).to(top_ckpts[0][1][k].dtype) for k in top_ckpts[0][1]}
    cur_m = model.module if hasattr(model, 'module') else model
    cur_m.load_state_dict(avg_state); model.eval()

    val_logits, test_logits = [], []
    with torch.no_grad():
        for batch in val_loader:
            inp = {k: v.to(DEVICE) for k, v in batch.items()}; inp.pop('labels', None)
            with autocast(enabled=USE_AMP): out = model(**inp)
            val_logits.extend(out.logits.float().cpu().numpy())
        for batch in test_loader:
            inp = {k: v.to(DEVICE) for k, v in batch.items()}
            with autocast(enabled=USE_AMP): out = model(**inp)
            test_logits.extend(out.logits.float().cpu().numpy())

    del model, optimizer, scheduler, scaler, top_ckpts, avg_state
    gc.collect(); torch.cuda.empty_cache()
    return np.array(val_logits), np.array(test_logits), best_f1


skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
mb_oof_logits = np.zeros((n_train, NUM_LABELS))
mb_test_sum   = np.zeros((n_test,  NUM_LABELS))
mb_fold_f1    = []

for fi, (tr, va) in enumerate(skf.split(np.zeros(n_train), y_all)):
    vl, tl, bf = train_fold_mb(fi, tr, va)
    mb_oof_logits[va] = vl; mb_test_sum += tl; mb_fold_f1.append(bf)

mb_test_logits = mb_test_sum / N_FOLDS

# Apply temperature (conservative [0.5, 2.0])
logits_t = torch.tensor(mb_oof_logits, dtype=torch.float32)
labels_t = torch.tensor(y_all, dtype=torch.long)
best_temp, best_nll = 1.0, float('inf')
for temp in np.arange(0.5, 2.0, 0.05):
    nll = F.cross_entropy(logits_t / temp, labels_t).item()
    if nll < best_nll: best_nll = nll; best_temp = temp

mb_oof_probs  = torch.softmax(logits_t / best_temp, dim=-1).numpy()
mb_test_probs = torch.softmax(torch.tensor(mb_test_logits, dtype=torch.float32) / best_temp, dim=-1).numpy()

mb_oof_f1 = f1_score(y_all, mb_oof_probs.argmax(1), average='macro')
print(f'\nModernBERT OOF F1: {mb_oof_f1:.4f}  (T={best_temp:.2f})')
print(f'Fold scores: {[f"{s:.4f}" for s in mb_fold_f1]}')


In [ ]:
# ============================================================
# STAGE 2: SVC on TF-IDF → OOF probs (5-fold)
# ============================================================
print('Building TF-IDF + SVC...')
char_tfidf = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5),
                              max_features=50000, min_df=2, max_df=0.95, sublinear_tf=True)
word_tfidf = TfidfVectorizer(analyzer='word',    ngram_range=(1, 2),
                              max_features=50000, min_df=2, max_df=0.95, sublinear_tf=True)
char_train = char_tfidf.fit_transform(texts_train); char_test = char_tfidf.transform(texts_test)
word_train = word_tfidf.fit_transform(texts_train); word_test = word_tfidf.transform(texts_test)
X_sp_train = sparse_hstack([char_train, word_train])
X_sp_test  = sparse_hstack([char_test,  word_test])

svc_oof  = np.zeros((n_train, NUM_LABELS))
svc_test = np.zeros((n_test,  NUM_LABELS))
skf5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for fi, (tr, va) in enumerate(skf5.split(np.zeros(n_train), y_all)):
    clf = CalibratedClassifierCV(
        LinearSVC(C=5.0, class_weight='balanced', max_iter=20000), cv=3, method='sigmoid')
    clf.fit(X_sp_train[tr], y_all[tr])
    svc_oof[va] = clf.predict_proba(X_sp_train[va])
    svc_test   += clf.predict_proba(X_sp_test) / 5
    print(f'  SVC fold {fi+1}: F1={f1_score(y_all[va], clf.predict(X_sp_train[va]), average="macro"):.4f}')

svc_f1 = f1_score(y_all, svc_oof.argmax(1), average='macro')
print(f'SVC OOF F1: {svc_f1:.4f}')


In [ ]:
# ============================================================
# STAGE 3: Stylometric features
# ============================================================
def extract_stylometric(texts):
    features = []
    for text in texts:
        chars   = len(text); words = text.split(); n_words = len(words)
        sents   = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]
        n_sents = max(len(sents), 1)
        sent_lens = [len(s.split()) for s in sents]
        n_para  = max(text.count('\n\n'), 1)
        def d(s): return text.count(s) / max(chars, 1)
        features.append([
            np.log1p(chars), np.log1p(n_words), np.log1p(n_sents), np.log1p(n_para),
            np.mean([len(w) for w in words]) if words else 0,
            n_words / n_sents,
            np.std(sent_lens) if len(sent_lens) > 1 else 0,
            n_words / n_para,
            len(set(w.lower() for w in words)) / max(n_words, 1),
            sum(c.isupper() for c in text) / max(chars, 1),
            sum(c.isdigit() for c in text) / max(chars, 1),
            d(','), d('.'), d('!'), d('?'), d(':'), d(';'),
            d('(') + d(')'), d('"') + d("'"), d('-') + d('\u2014'), d('\n'),
        ])
    return np.array(features, dtype=np.float32)

print('Extracting stylometric features...')
X_style_train = extract_stylometric(texts_train)
X_style_test  = extract_stylometric(texts_test)
scaler = StandardScaler()
X_style_train = scaler.fit_transform(X_style_train)
X_style_test  = scaler.transform(X_style_test)
print(f'Stylometric shape: {X_style_train.shape}')


In [ ]:
# ============================================================
# STAGE 4: LightGBM meta-learner on stacked features
# Input: [MB_probs(6) + SVC_probs(6) + stylometric(21)] = 33 features
# Trained with 5-fold CV to get meta-OOF predictions
# ============================================================
meta_train = np.hstack([mb_oof_probs, svc_oof, X_style_train])   # (2400, 33)
meta_test  = np.hstack([mb_test_probs, svc_test, X_style_test])  # (600, 33)

print(f'Meta-learner input shape: train={meta_train.shape}, test={meta_test.shape}')

meta_oof     = np.zeros((n_train, NUM_LABELS))
meta_test_sum = np.zeros((n_test,  NUM_LABELS))

lgb_params = dict(
    n_estimators=1000, learning_rate=0.05, num_leaves=31,
    min_child_samples=10, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=1.0, reg_lambda=1.0,   # stronger regularization: only 2400 samples
    class_weight='balanced',
    random_state=SEED, n_jobs=-1, verbose=-1
)

print('Training LightGBM meta-learner (5-fold)...')
meta_fold_f1 = []
for fi, (tr, va) in enumerate(skf5.split(np.zeros(n_train), y_all)):
    meta = lgb.LGBMClassifier(**lgb_params)
    meta.fit(
        meta_train[tr], y_all[tr],
        eval_set=[(meta_train[va], y_all[va])],
        callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
    meta_oof[va] = meta.predict_proba(meta_train[va])
    meta_test_sum += meta.predict_proba(meta_test) / 5
    f1 = f1_score(y_all[va], meta_oof[va].argmax(1), average='macro')
    meta_fold_f1.append(f1)
    print(f'  Meta fold {fi+1}: F1={f1:.4f}  (best_iter={meta.best_iteration_})')

meta_f1 = f1_score(y_all, meta_oof.argmax(1), average='macro')
print(f'\nMeta-LGB OOF F1: {meta_f1:.4f}')
print(f'Fold scores:     {[f"{s:.4f}" for s in meta_fold_f1]}')
print()
print(classification_report(y_all, meta_oof.argmax(1),
      target_names=[LABEL_MAP[i] for i in range(NUM_LABELS)]))


In [ ]:
# ============================================================
# FINAL PREDICTIONS
# No threshold sweep needed — meta-learner handles class imbalance directly
# ============================================================
final_preds = meta_test_sum.argmax(1)

id_col = None
for c in test_df.columns:
    if c.lower() in ('id', 'unnamed: 0', 'index'): id_col = c; break

pred_labels = [LABEL_MAP[p] for p in final_preds]
submission  = pd.DataFrame({'id': test_df[id_col], 'LABEL': pred_labels}) if id_col \
              else pd.DataFrame({'LABEL': pred_labels})
submission.to_csv('submission.csv', index=False)

print('Prediction distribution:')
for i, nm in LABEL_MAP.items():
    cnt = (final_preds == i).sum()
    print(f'  {nm:10s}: {cnt} ({cnt/len(final_preds)*100:.1f}%)')

print(f'\n{"="*50}')
print(f'FINAL SUMMARY (v8 — LightGBM Stacking)')
print(f'  ModernBERT OOF F1:   {mb_oof_f1:.4f}')
print(f'  SVC OOF F1:          {svc_f1:.4f}')
print(f'  Meta-LGB OOF F1:     {meta_f1:.4f}')
print(f'  No threshold sweep   (meta-learner replaces it)')
print(f'  DRW cap: {DRW_CAP}x  |  MB folds: {N_FOLDS}  |  Meta folds: 5')
print(f'  Total time:          {(time.time()-t_start)/60:.1f} min')
print(f'{"="*50}')
print('Submission saved!')
